In [1]:
import pandas as pd
import sqlite3
import os
import warnings
warnings.filterwarnings('ignore')

# Load cleaned data
df_b = pd.read_csv('../data/processed/bengaluru_cleaned.csv')
df_g = pd.read_csv('../data/processed/global_cleaned.csv')

# Create SQLite database
os.makedirs('../data', exist_ok=True)
conn = sqlite3.connect('../data/zomato.db')

# Push to SQL tables
df_b.to_sql('bengaluru',    conn, if_exists='replace', index=False)
df_g.to_sql('global_india', conn, if_exists='replace', index=False)

print("✅ Database created successfully")
print(f"bengaluru    → {len(df_b):,} rows")
print(f"global_india → {len(df_g):,} rows")

# Helper function
def run_query(query, label=""):
    result = pd.read_sql_query(query, conn)
    if label:
        print(f"\n{'='*55}")
        print(f"  {label}")
        print(f"{'='*55}")
    print(result.to_string(index=False))
    return result

print("\n✅ run_query function ready!")

✅ Database created successfully
bengaluru    → 41,410 rows
global_india → 6,513 rows

✅ run_query function ready!


In [2]:
run_query("""
    SELECT listed_type,
           COUNT(*) AS total_restaurants,
           ROUND(AVG(rate), 2) AS avg_rating,
           ROUND(AVG(cost_for_two), 2) AS avg_cost
    FROM bengaluru
    GROUP BY listed_type
    ORDER BY total_restaurants DESC
""", "Q1: Restaurant Type Distribution")


  Q1: Restaurant Type Distribution
       listed_type  total_restaurants  avg_rating  avg_cost
          Delivery              20534        3.65    495.21
          Dine-out              14126        3.68    654.57
          Desserts               2714        3.78    408.89
             Cafes               1511        3.87    646.43
Drinks & nightlife               1045        4.02   1454.78
            Buffet                848        3.98   1316.63
     Pubs and bars                632        4.02   1336.71


,listed_type,total_restaurants,avg_rating,avg_cost
0,Delivery,20534,3.65,495.21
1,Dine-out,14126,3.68,654.57
2,Desserts,2714,3.78,408.89
3,Cafes,1511,3.87,646.43
4,Drinks & nightlife,1045,4.02,1454.78
5,Buffet,848,3.98,1316.63
6,Pubs and bars,632,4.02,1336.71


In [3]:
run_query("""
    SELECT location,
           COUNT(*) AS total_restaurants,
           ROUND(AVG(rate), 2) AS avg_rating,
           ROUND(AVG(cost_for_two), 2) AS avg_cost,
           ROUND(AVG(votes), 0) AS avg_votes
    FROM bengaluru
    GROUP BY location
    HAVING total_restaurants >= 50
    ORDER BY avg_rating DESC
    LIMIT 10
""", "Q2: Top 10 Locations by Rating")


  Q2: Top 10 Locations by Rating
             location  total_restaurants  avg_rating  avg_cost  avg_votes
         Lavelle Road                481        4.14   1365.38     1051.0
       St. Marks Road                343        4.02    883.67      776.0
Koramangala 3rd Block                191        4.02    834.82      655.0
Koramangala 5th Block               2297        4.01    680.71      964.0
        Church Street                546        3.99    839.84     1090.0
Koramangala 4th Block                841        3.92    758.32      815.0
      Cunningham Road                475        3.90    867.16      606.0
       Residency Road                604        3.86   1030.05      483.0
              MG Road                793        3.85   1244.51      540.0
Koramangala 7th Block               1055        3.85    604.36      469.0


,location,total_restaurants,avg_rating,avg_cost,avg_votes
0,Lavelle Road,481,4.14,1365.38,1051.0
1,St. Marks Road,343,4.02,883.67,776.0
2,Koramangala 3rd Block,191,4.02,834.82,655.0
3,Koramangala 5th Block,2297,4.01,680.71,964.0
4,Church Street,546,3.99,839.84,1090.0
5,Koramangala 4th Block,841,3.92,758.32,815.0
6,Cunningham Road,475,3.90,867.16,606.0
7,Residency Road,604,3.86,1030.05,483.0
8,MG Road,793,3.85,1244.51,540.0
9,Koramangala 7th Block,1055,3.85,604.36,469.0


In [4]:
run_query("""
    SELECT
        CASE WHEN online_order = 1
             THEN 'Has Online Order'
             ELSE 'No Online Order'
        END AS order_type,
        COUNT(*) AS total,
        ROUND(AVG(rate), 2) AS avg_rating,
        ROUND(AVG(cost_for_two), 2) AS avg_cost,
        ROUND(AVG(votes), 2) AS avg_votes
    FROM bengaluru
    GROUP BY online_order
""", "Q3: Online Order Impact on Rating")


  Q3: Online Order Impact on Rating
      order_type  total  avg_rating  avg_cost  avg_votes
 No Online Order  14212        3.66    716.03     367.99
Has Online Order  27198        3.72    544.42     343.33


,order_type,total,avg_rating,avg_cost,avg_votes
0,No Online Order,14212,3.66,716.03,367.99
1,Has Online Order,27198,3.72,544.42,343.33


In [5]:
run_query("""
    SELECT
        CASE WHEN book_table = 1
             THEN 'Table Booking'
             ELSE 'No Booking'
        END AS booking_type,
        COUNT(*) AS total,
        ROUND(AVG(rate), 2) AS avg_rating,
        ROUND(AVG(cost_for_two), 2) AS avg_cost,
        ROUND(AVG(votes), 2) AS avg_votes,
        SUM(CASE WHEN rate >= 4.0
                 THEN 1 ELSE 0 END) AS high_rated_count
    FROM bengaluru
    GROUP BY book_table
""", "Q4: Table Booking Premium Analysis")


  Q4: Table Booking Premium Analysis
 booking_type  total  avg_rating  avg_cost  avg_votes  high_rated_count
   No Booking  35106        3.62    482.43     204.62              7389
Table Booking   6304        4.14   1276.49    1171.34              4918


,booking_type,total,avg_rating,avg_cost,avg_votes,high_rated_count
0,No Booking,35106,3.62,482.43,204.62,7389
1,Table Booking,6304,4.14,1276.49,1171.34,4918


In [6]:
run_query("""
    SELECT
        TRIM(SUBSTR(cuisines, 1,
             INSTR(cuisines || ',', ',') - 1)) AS primary_cuisine,
        COUNT(*) AS total_restaurants,
        ROUND(AVG(rate), 2) AS avg_rating,
        ROUND(AVG(cost_for_two), 2) AS avg_cost,
        ROUND(AVG(votes), 2) AS avg_votes
    FROM bengaluru
    GROUP BY primary_cuisine
    HAVING total_restaurants >= 100
    ORDER BY avg_rating DESC
    LIMIT 10
""", "Q5: Top Cuisines by Performance")


  Q5: Top Cuisines by Performance
primary_cuisine  total_restaurants  avg_rating  avg_cost  avg_votes
  Modern Indian                109        4.31   1585.32    1080.38
       European                229        4.26   1790.39    1637.94
  Mediterranean                101        4.20   1355.45    1356.23
       Japanese                112        4.19   1937.50     918.21
       American                441        4.16   1352.95    1906.46
          Asian                370        4.15   1327.57     920.62
           Thai                128        4.04   1473.05     668.32
            BBQ                108        4.00    897.69     452.10
    Continental               1638        3.98   1155.83     976.06
     Rajasthani                110        3.96    730.91     693.63


,primary_cuisine,total_restaurants,avg_rating,avg_cost,avg_votes
0,Modern Indian,109,4.31,1585.32,1080.38
1,European,229,4.26,1790.39,1637.94
2,Mediterranean,101,4.20,1355.45,1356.23
3,Japanese,112,4.19,1937.50,918.21
4,American,441,4.16,1352.95,1906.46
5,Asian,370,4.15,1327.57,920.62
6,Thai,128,4.04,1473.05,668.32
7,BBQ,108,4.00,897.69,452.10
8,Continental,1638,3.98,1155.83,976.06
9,Rajasthani,110,3.96,730.91,693.63


In [7]:
run_query("""
    SELECT
        location,
        COUNT(*) AS restaurant_count,
        ROUND(AVG(votes), 0) AS avg_votes,
        ROUND(AVG(rate), 2) AS avg_rating,
        ROUND(CAST(SUM(votes) AS FLOAT) / COUNT(*), 2)
            AS demand_supply_ratio
    FROM bengaluru
    GROUP BY location
    HAVING restaurant_count >= 30
    ORDER BY demand_supply_ratio DESC
    LIMIT 10
""", "Q6: High Demand Low Supply Locations")


  Q6: High Demand Low Supply Locations
             location  restaurant_count  avg_votes  avg_rating  demand_supply_ratio
        Church Street               546     1090.0        3.99              1089.71
         Lavelle Road               481     1051.0        4.14              1050.85
Koramangala 5th Block              2297      964.0        4.01               964.23
Koramangala 4th Block               841      815.0        3.92               814.69
       St. Marks Road               343      776.0        4.02               775.80
Koramangala 3rd Block               191      655.0        4.02               655.28
          Indiranagar              1803      651.0        3.83               650.50
      Cunningham Road               475      606.0        3.90               606.05
              MG Road               793      540.0        3.85               540.34
       Residency Road               604      483.0        3.86               483.35


,location,restaurant_count,avg_votes,avg_rating,demand_supply_ratio
0,Church Street,546,1090.0,3.99,1089.71
1,Lavelle Road,481,1051.0,4.14,1050.85
2,Koramangala 5th Block,2297,964.0,4.01,964.23
3,Koramangala 4th Block,841,815.0,3.92,814.69
4,St. Marks Road,343,776.0,4.02,775.80
5,Koramangala 3rd Block,191,655.0,4.02,655.28
6,Indiranagar,1803,651.0,3.83,650.50
7,Cunningham Road,475,606.0,3.90,606.05
8,MG Road,793,540.0,3.85,540.34
9,Residency Road,604,483.0,3.86,483.35


In [8]:
run_query("""
    SELECT
        rest_type,
        COUNT(*) AS total,
        ROUND(AVG(rate), 2) AS avg_rating,
        ROUND(AVG(cost_for_two), 2) AS avg_cost,
        SUM(CASE WHEN rate >= 4.0
                 THEN 1 ELSE 0 END) AS high_rated_count,
        ROUND(
            CAST(SUM(CASE WHEN rate >= 4.0
                          THEN 1 ELSE 0 END) AS FLOAT)
            / COUNT(*) * 100, 1
        ) AS pct_high_rated
    FROM bengaluru
    WHERE rest_type IS NOT NULL
    GROUP BY rest_type
    HAVING total >= 100
    ORDER BY avg_rating DESC
    LIMIT 10
""", "Q7: Restaurant Type Performance")


  Q7: Restaurant Type Performance
                  rest_type  total  avg_rating  avg_cost  high_rated_count  pct_high_rated
Microbrewery, Casual Dining    121        4.37   1676.03               120            99.2
        Casual Dining, Cafe    310        4.20    921.29               260            83.9
         Casual Dining, Pub    127        4.18   1320.47               119            93.7
                Fine Dining    343        4.15   2720.70               268            78.1
       Cafe, Dessert Parlor    144        4.15    670.83               117            81.3
       Dessert Parlor, Cafe    144        4.14    550.00               111            77.1
         Bar, Casual Dining    385        4.13   1382.34               279            72.5
         Pub, Casual Dining    236        4.09   1247.03               167            70.8
         Casual Dining, Bar   1092        4.08   1261.08               795            72.8
        Cafe, Casual Dining    173        4.05    815.3

,rest_type,total,avg_rating,avg_cost,high_rated_count,pct_high_rated
0,"Microbrewery, Casual Dining",121,4.37,1676.03,120,99.2
1,"Casual Dining, Cafe",310,4.20,921.29,260,83.9
2,"Casual Dining, Pub",127,4.18,1320.47,119,93.7
3,Fine Dining,343,4.15,2720.70,268,78.1
4,"Cafe, Dessert Parlor",144,4.15,670.83,117,81.3
5,"Dessert Parlor, Cafe",144,4.14,550.00,111,77.1
6,"Bar, Casual Dining",385,4.13,1382.34,279,72.5
7,"Pub, Casual Dining",236,4.09,1247.03,167,70.8
8,"Casual Dining, Bar",1092,4.08,1261.08,795,72.8
9,"Cafe, Casual Dining",173,4.05,815.32,97,56.1


In [9]:
run_query("""
    SELECT
        location,
        COUNT(*) AS restaurant_count,
        ROUND(AVG(rate), 2) AS avg_rating,
        ROUND(AVG(cost_for_two), 2) AS avg_cost,
        SUM(votes) AS total_votes,
        ROUND(CAST(SUM(votes) AS FLOAT) / COUNT(*), 0)
            AS votes_per_restaurant,
        CASE
            WHEN CAST(SUM(votes) AS FLOAT)/COUNT(*) > 400
             AND COUNT(*) < 500
             THEN 'HIGH PRIORITY'
            WHEN CAST(SUM(votes) AS FLOAT)/COUNT(*) > 200
             AND COUNT(*) < 500
             THEN 'MEDIUM PRIORITY'
            ELSE 'SATURATED'
        END AS expansion_priority
    FROM bengaluru
    GROUP BY location
    HAVING restaurant_count >= 30
    ORDER BY votes_per_restaurant DESC
    LIMIT 15
""", "Q8: EXPANSION RECOMMENDATION")


  Q8: EXPANSION RECOMMENDATION
             location  restaurant_count  avg_rating  avg_cost  total_votes  votes_per_restaurant expansion_priority
        Church Street               546        3.99    839.84       594979                1090.0          SATURATED
         Lavelle Road               481        4.14   1365.38       505460                1051.0      HIGH PRIORITY
Koramangala 5th Block              2297        4.01    680.71      2214827                 964.0          SATURATED
Koramangala 4th Block               841        3.92    758.32       685156                 815.0          SATURATED
       St. Marks Road               343        4.02    883.67       266099                 776.0      HIGH PRIORITY
Koramangala 3rd Block               191        4.02    834.82       125159                 655.0      HIGH PRIORITY
          Indiranagar              1803        3.83    679.00      1172855                 651.0          SATURATED
      Cunningham Road               475 

,location,restaurant_count,avg_rating,avg_cost,total_votes,votes_per_restaurant,expansion_priority
0,Church Street,546,3.99,839.84,594979,1090.0,SATURATED
1,Lavelle Road,481,4.14,1365.38,505460,1051.0,HIGH PRIORITY
2,Koramangala 5th Block,2297,4.01,680.71,2214827,964.0,SATURATED
3,Koramangala 4th Block,841,3.92,758.32,685156,815.0,SATURATED
4,St. Marks Road,343,4.02,883.67,266099,776.0,HIGH PRIORITY
5,Koramangala 3rd Block,191,4.02,834.82,125159,655.0,HIGH PRIORITY
6,Indiranagar,1803,3.83,679.00,1172855,651.0,SATURATED
7,Cunningham Road,475,3.90,867.16,287873,606.0,HIGH PRIORITY
8,MG Road,793,3.85,1244.51,428489,540.0,SATURATED
9,Residency Road,604,3.86,1030.05,291945,483.0,SATURATED


In [10]:
sql_content = """-- ================================================
-- Zomato Data Analysis - SQL Business Queries
-- Author: Yash Chavan | VIT Pune CSE-DS 2024-28
-- ================================================

-- Q1: Restaurant Type Distribution
SELECT listed_type,
       COUNT(*) AS total_restaurants,
       ROUND(AVG(rate), 2) AS avg_rating
FROM bengaluru
GROUP BY listed_type
ORDER BY total_restaurants DESC;

-- Q2: Top 10 Locations by Rating
SELECT location,
       COUNT(*) AS total_restaurants,
       ROUND(AVG(rate), 2) AS avg_rating
FROM bengaluru
GROUP BY location
HAVING total_restaurants >= 50
ORDER BY avg_rating DESC
LIMIT 10;

-- Q3: Online Order Impact
SELECT CASE WHEN online_order = 1
            THEN 'Has Online Order'
            ELSE 'No Online Order' END AS order_type,
       COUNT(*) AS total,
       ROUND(AVG(rate), 2) AS avg_rating
FROM bengaluru
GROUP BY online_order;

-- Q4: Table Booking Analysis
SELECT CASE WHEN book_table = 1
            THEN 'Table Booking'
            ELSE 'No Booking' END AS booking_type,
       COUNT(*) AS total,
       ROUND(AVG(rate), 2) AS avg_rating,
       ROUND(AVG(cost_for_two), 2) AS avg_cost
FROM bengaluru
GROUP BY book_table;

-- Q5: Cuisine Performance
SELECT TRIM(SUBSTR(cuisines, 1,
       INSTR(cuisines || ',', ',') - 1)) AS primary_cuisine,
       COUNT(*) AS total,
       ROUND(AVG(rate), 2) AS avg_rating
FROM bengaluru
GROUP BY primary_cuisine
HAVING total >= 100
ORDER BY avg_rating DESC
LIMIT 10;

-- Q6: Expansion Recommendation
SELECT location,
       COUNT(*) AS restaurant_count,
       ROUND(CAST(SUM(votes) AS FLOAT)/COUNT(*), 0)
           AS votes_per_restaurant,
       ROUND(AVG(rate), 2) AS avg_rating
FROM bengaluru
GROUP BY location
HAVING restaurant_count >= 30
ORDER BY votes_per_restaurant DESC
LIMIT 15;
"""

os.makedirs('../sql', exist_ok=True)
with open('../sql/business_queries.sql', 'w', encoding='utf-8') as f:
    f.write(sql_content)

conn.close()

print("✅ SQL file saved → sql/business_queries.sql")
print("✅ Database closed")
print("✅ Phase 6 COMPLETE!")

✅ SQL file saved → sql/business_queries.sql
✅ Database closed
✅ Phase 6 COMPLETE!
